# Task 4 — NOVA brand voice (QLoRA)

**Runtime:** Google Colab **GPU** (`Runtime → Change runtime type → T4` or better). QLoRA needs CUDA + `bitsandbytes`.

**Secrets (Colab):** `WANDB_API_KEY` (optional, for dashboards), `HF_TOKEN` (optional, to push the adapter).

**Default model:** `TinyLlama/TinyLlama-1.1B-Chat-v1.0` (ungated). Swap `MODEL_ID` for `meta-llama/Llama-3.2-1B-Instruct` if you have HF access.

After training, add your **public W&B run** and **HF model** URLs to [`README.md`](README.md).

In [ ]:
# Colab / local: install stack (re-run after new runtime)
%pip install -q "transformers>=4.46.0,<5" "peft>=0.11.0" accelerate bitsandbytes "trl>=0.9.0" wandb datasets sentencepiece protobuf

In [ ]:
import os
import torch
from datasets import Dataset

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU runtime (Colab: T4+). CPU training with 4-bit is not supported here.")

DEVICE = "cuda"
print("GPU:", torch.cuda.get_device_name(0))

# Optional: Colab secrets → os.environ (uncomment and match your secret names)
# from google.colab import userdata
# os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
# os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

REPORT_TO = "wandb" if os.environ.get("WANDB_API_KEY") else "none"
if REPORT_TO == "wandb":
    import wandb

    wandb.login(key=os.environ["WANDB_API_KEY"], relogin=True)
    wandb.init(project="nova-brand-voice", name="qlora-nova-task4")
    print("W&B logging enabled.")
else:
    print("W&B disabled (set WANDB_API_KEY to log a public run for submission).")

In [ ]:
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
OUTPUT_DIR = "./nova_brand_lora_adapter"
SYSTEM_PROMPT = (
    "You are NOVA's customer care assistant for a D2C fashion and beauty brand. "
    "Reply in a warm, concise, inclusive tone. Never invent order numbers or policies."
)

In [ ]:
# Minimal synthetic brand-voice dataset (expand or swap for CSV / HF dataset)
brand_samples = [
    {"instruction": "Customer asks about a delayed order.", "response": "Hi love — I'm so sorry for the wait. I've pulled up your tracking and I'm here to make this right."},
    {"instruction": "Is the Velvet Lip Tint drying?", "response": "Great question! It's long-wear, but we balanced it with oils so it stays comfy on the lips."},
    {"instruction": "Customer is upset about the wrong shade.", "response": "I hear you — that’s frustrating. Let’s swap it quickly; I’ll set up a return label for you."},
    {"instruction": "Do you ship to Canada?", "response": "Yes — we ship to Canada. Duties and taxes may apply at delivery; I can link you to our shipping FAQ."},
    {"instruction": "How do I start a return?", "response": "Happy to help. Open Orders → Return item, or tell me your order ID and I’ll walk you through it."},
    {"instruction": "Is the HydraCalm serum OK for sensitive skin?", "response": "It’s formulated fragrance-free with barrier-friendly ingredients — patch test 24h if your skin is reactive."},
    {"instruction": "Customer says the package arrived damaged.", "response": "I'm really sorry — that’s not the experience we want. Send a quick photo if you can; we’ll replace or refund."},
    {"instruction": "What’s your clean beauty stance?", "response": "We prioritize transparent ingredient lists and skip a set of harsh additives on our Clean Edit — happy to point you to specifics."},
    {"instruction": "Jeans sizing — should I size up?", "response": "This cut has a touch of stretch; most guests stay true to size. If you’re between sizes, I’d try the smaller first."},
    {"instruction": "Customer wants a product recommendation for dry winter skin.", "response": "For winter, I’d layer a gentle cleanser, a ceramide-rich serum, then a richer cream — want me to match NOVA SKUs?"},
    {"instruction": "Can I change my order before it ships?", "response": "If it’s still processing, we can usually edit it — share your order ID and what you’d like to change."},
    {"instruction": "Customer is angry and uses all caps.", "response": "I’m really sorry this has been stressful. I’m on it — give me one moment to pull up your details so we can fix this."},
    {"instruction": "Reward points didn’t apply.", "response": "Let me check your rewards balance and that checkout — sometimes codes stack differently. I’ll sort it out with you."},
    {"instruction": "Vegan options for lip products?", "response": "I can filter our vegan lip lineup — a few hero shades are fully vegan; want cool or warm undertones?"},
    {"instruction": "How long does standard shipping take?", "response": "Typically 3–5 business days domestically once it leaves our warehouse — I’ll confirm with your ZIP if you’d like."},
    {"instruction": "Customer asks for a discount.", "response": "I can’t always promise codes, but I’ll check what’s active on your account and any eligible promos right now."},
    {"instruction": "Sunscreen under makeup — which NOVA SPF?", "response": "Our fluid SPF layers well under makeup — use two fingers’ worth for face and let it set 60s before base."},
    {"instruction": "Hair mask — how often?", "response": "For most hair types, 1–2× weekly is plenty; if hair is very dry, you can build up slowly and rinse well."},
    {"instruction": "Gift message at checkout?", "response": "Yes — add your note in the gift message box; we’ll print it on the packing slip without showing prices."},
    {"instruction": "Customer says they’re gifting and nervous about shade.", "response": "Totally get it — gift receipts make exchanges easy, and I can suggest a universally flattering neutral."},
    {"instruction": "Ingredient question: retinol with your serum?", "response": "If you’re new to retinol, alternate nights at first and always SPF in the morning — want me to check compatibility with your cart?"},
    {"instruction": "Order shows delivered but customer didn’t get it.", "response": "That’s worrying — let’s verify the address and carrier photo. I can open a trace and backup options if needed."},
    {"instruction": "Do you have inclusive shade ranges?", "response": "Yes — we’re expanding depth each season; tell me your undertone and coverage preference and I’ll narrow it down."},
    {"instruction": "Customer thanks you warmly.", "response": "That means a lot — thank you for being part of NOVA. We're here anytime."},
]
len(brand_samples)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


def row_to_text(example: dict) -> dict:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["response"]},
    ]
    # TinyLlama chat template supports system/user/assistant
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}


ds = Dataset.from_list(brand_samples)
ds = ds.map(row_to_text, remove_columns=[c for c in ds.column_names if c != "text"])
ds = ds.train_test_split(test_size=0.12, seed=42)
train_ds, eval_ds = ds["train"], ds["test"]
train_ds[0]["text"][:200]

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)
model

In [ ]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_ratio=0.08,
    logging_steps=1,
    eval_strategy="steps",
    eval_steps=10,
    save_strategy="epoch",
    fp16=True,
    bf16=False,
    optim="paged_adamw_8bit",
    report_to=REPORT_TO,
    dataset_text_field="text",
    max_seq_length=512,
    gradient_checkpointing=True,
    load_best_model_at_end=False,
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
    peft_config=peft_config,
)

trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved adapter to", OUTPUT_DIR)

## Brand-voice spot check

Qualitative rubric (for screenshots / report): **warmth**, **conciseness**, **on-brand vocabulary**, **no fabricated order data**.

Compare the same prompts before and after fine-tuning if you keep a base checkpoint; here we show **adapter** generations only.

In [ ]:
from peft import PeftModel

torch.cuda.empty_cache()

base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    device_map="auto",
    trust_remote_code=True,
)
ft = PeftModel.from_pretrained(base, OUTPUT_DIR)
ft.eval()


@torch.inference_mode()
def generate_answer(instruction: str, max_new_tokens: int = 160) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": instruction},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(ft.device)
    out = ft.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.65,
        top_p=0.9,
        pad_token_id=tokenizer.pad_token_id,
    )
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    # Return assistant tail only
    if "<|assistant|>" in text:
        return text.split("<|assistant|>")[-1].strip()
    return text[len(prompt) :].strip()


for q in [
    "My order is late and I'm annoyed.",
    "Which NOVA lip product won't dry me out?",
    "I need help with a return — wrong size.",
]:
    print("---")
    print("Q:", q)
    print("A:", generate_answer(q))

## Optional: push adapter to Hugging Face Hub

1. Create a model repo (e.g. `yourname/nova-brand-voice-tinyllama-lora`).
2. Set `HF_TOKEN` (write access).
3. Run the cell below.

In [ ]:
HF_REPO = os.environ.get("HF_REPO", "YOUR_USERNAME/nova-brand-voice-lora")  # change me

if os.environ.get("HF_TOKEN") and not HF_REPO.startswith("YOUR_"):
    from huggingface_hub import login

    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
    ft.push_to_hub(HF_REPO, token=os.environ["HF_TOKEN"])
    tokenizer.push_to_hub(HF_REPO, token=os.environ["HF_TOKEN"])
    print("Pushed to https://huggingface.co/", HF_REPO, sep="")
else:
    print("Skip push: set HF_TOKEN and HF_REPO to your model id.")

In [ ]:
if REPORT_TO == "wandb":
    wandb.finish()
    print("W&B run finished — copy the public URL from the W&B UI for README.md.")